In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import re, math
import time
import pandas as pd
import os
from time import sleep
from io import BytesIO


## Scrape

In [2]:
BASE_URL = "https://espc.com/properties?ps=50&locations=edinburgh&minbeds=2plus&page={}"
url = "https://espc.com/properties?ps=50&locations=edinburgh&minbeds=2plus"
HEADERS = {"User-Agent": "Mozilla/5.0"}

POSTCODE_PATTERN = re.compile(
    r"\b[A-Z]{1,2}\d{1,2}[A-Z]?\s*\d[A-Z]{2}\b",
    re.IGNORECASE
)


In [5]:
page = 1

url = f"{BASE_URL}&page={page}"
response = requests.get(url, headers=HEADERS)
soup = BeautifulSoup(response.text, "html.parser")

results = soup.select_one(".results-per-page")
text = results.get_text(" ", strip=True)

numbers = re.findall(r"\d+", text)

per_page = int(numbers[1])
total_properties = int(numbers[2])

total_pages = math.ceil(total_properties / per_page)

print(f"Total properties: {total_properties}")
print(f"Properties per page: {per_page}")
print(f"Total pages: {total_pages}")

Total properties: 984
Properties per page: 50
Total pages: 20


In [48]:
for page in range(1, 2):
    url = f"{BASE_URL}&page={page}"
    response = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")
    cards = soup.select(".propertyWrap, .property-card, article")
    print(f"Page {page}: {len(cards)} listings")

Page 1: 53 listings


In [6]:
# function to fetch a page and return the BeautifulSoup object

def fetch_page(page):
    url = BASE_URL.format(page)
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def extract_postcode(address):
    match = POSTCODE_PATTERN.search(address or "")
    return match.group().upper() if match else None


def extract_facilities(card):
    # Try facilities/features blocks first
    for selector in [".facilities span", ".property__features li"]:
        values = [
            int(x.get_text(strip=True))
            for x in card.select(selector)
            if x.get_text(strip=True).isdigit()
        ]

        if len(values) >= 3:
            return values[:3]

    # Fallback
    text = card.get_text(" ", strip=True).lower()

    bed = re.search(r"(\d+)\s*bed", text)
    bath = re.search(r"(\d+)\s*bath", text)
    reception = re.search(r"(\d+)\s*reception", text)

    return (
        int(bed.group(1)) if bed else None,
        int(bath.group(1)) if bath else None,
        int(reception.group(1)) if reception else None
    )


def parse_listing(card):
    text = card.get_text(" ", strip=True)

    # Property type and address
    prop_type, address = (
        [x.strip() for x in text.split(":", 1)]
        if ":" in text
        else (None, text)
    )

    # Price
    price_text = card.find(string=lambda x: x and "£" in x)
    price_match = re.search(r"£([\d,]+)", price_text or "")
    price = int(price_match.group(1).replace(",", "")) if price_match else None

    # Facilities
    beds, baths, receptions = extract_facilities(card)

    # URL
    link = card.find("a", href=True)
    url = f"https://espc.com{link['href']}" if link else None

    # Short description
    desc = card.select_one(".description, .itemDescription")

    return {
        "property_type": prop_type,
        "address": address,
        "postcode": extract_postcode(address),
        "price": price,
        "beds": beds,
        "baths": baths,
        "receptions": receptions,
        "url": url,
        "search_description": desc.get_text(" ", strip=True) if desc else None
    }


def scrape_espc(total_pages):
    properties = []

    for page in range(1, total_pages + 1):
        soup = fetch_page(page)
        cards = soup.select(".propertyWrap, .property-card, article")

        print(f"Page {page}/{total_pages}: {len(cards)} listings")

        for card in cards:
            item = parse_listing(card)

            if (
                item["beds"] is not None
                and item["baths"] is not None
                and item["price"] is not None
                and item["beds"] >= 2
                and item["baths"] >= 2
                and 200000 <= item["price"] <= 350000
            ):
                properties.append(item)

        time.sleep(1)

    return pd.DataFrame(properties)

In [7]:
df = scrape_espc(total_pages)

print(f"Properties found: {len(df)}")
df.head()

Page 1/20: 53 listings
Page 2/20: 53 listings
Page 3/20: 53 listings
Page 4/20: 53 listings
Page 5/20: 53 listings
Page 6/20: 53 listings
Page 7/20: 53 listings
Page 8/20: 53 listings
Page 9/20: 53 listings
Page 10/20: 53 listings
Page 11/20: 53 listings
Page 12/20: 53 listings
Page 13/20: 53 listings
Page 14/20: 53 listings
Page 15/20: 53 listings
Page 16/20: 53 listings
Page 17/20: 53 listings
Page 18/20: 53 listings
Page 19/20: 53 listings
Page 20/20: 37 listings
Properties found: 168


,property_type,address,postcode,price,beds,baths,receptions,url,search_description
0,"New Exclusive Fixed Price £220,000 2 2 1 New E...","Flat 28 , 3 Salamander Court , Leith , EH6 7JE...",EH6 7JE,220000,2,2,1,https://espc.com/property/flat-28-3-salamander...,McEwan Fraser Legal is delighted to present th...
1,"New Exclusive Virtual Tour Offers Over £240,00...","17/6 King Street , Leith , Edinburgh , EH6 6TQ...",EH6 6TQ,240000,2,2,1,https://espc.com/property/17-6-king-street-lei...,"Forming part of an exclusive development, this..."
2,"New Exclusive Video Offers Over £340,000 2 2 2...","30/8 Blackwood Crescent , Newington , Edinburg...",EH9 1QX,340000,2,2,2,https://espc.com/property/30-8-blackwood-cresc...,"Simply stunning two-bedroom, double upper styl..."
3,"Featured Virtual Tour Offers Over £255,000 2 2...","Flat 9 , 5 Waterfront Avenue , Edinburgh , EH5...",EH5 1RT,255000,2,2,1,https://espc.com/property/flat-9-5-waterfront-...,Set on the first floor of a desirable resident...
4,"Video Offers Over £350,000 3 2 1 Video Ground ...","54 (flat 2) , Stanley Place , Abbeyhill , Edin...",EH7 5TB,350000,3,2,1,https://espc.com/property/54-flat-2-stanley-pl...,Seldom available 3 bed elevated ground floor a...


In [8]:
df.to_csv("espc_filtered.csv", index=False)

### old 

In [ ]:
def fetch_page(page_number):
    """Download one ESPC listing page."""
    url = BASE_URL.format(page_number)
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

def extract_postcode(address):
    """Extract full UK postcode."""
    if not address:
        return None
    m = POSTCODE_PATTERN.search(address)
    return m.group(0).upper() if m else None

def extract_facilities(card):
    """Extract beds, baths, receptions using multiple strategies."""

    # Strategy 1: facilities block
    fac = card.select_one(".facilities")
    if fac:
        nums = [
            int(x.get_text(strip=True))
            for x in fac.find_all("span")
            if x.get_text(strip=True).isdigit()
        ]
        if len(nums) >= 3:
            return nums[0], nums[1], nums[2]

    # Strategy 2: property__features list
    feats = card.select(".property__features li")
    if feats:
        nums = [
            int(x.get_text(strip=True))
            for x in feats
            if x.get_text(strip=True).isdigit()
        ]
        if len(nums) >= 3:
            return nums[0], nums[1], nums[2]

    # Strategy 3: labelled spans
    beds = baths = receptions = None
    for span in card.find_all("span"):
        txt = span.get_text(strip=True).lower()
        if "bed" in txt and txt.replace("bedrooms", "").strip().isdigit():
            beds = int(txt.replace("bedrooms", "").strip())
        if "bath" in txt and txt.replace("bathrooms", "").strip().isdigit():
            baths = int(txt.replace("bathrooms", "").strip())

    if beds and baths:
        return beds, baths, None

    # Strategy 4: fallback regex
    text = card.get_text(" ", strip=True)
    nums = re.findall(r"\b\d+\b", text)
    if len(nums) >= 3:
        return int(nums[0]), int(nums[1]), int(nums[2])

    return None, None, None


    """
    Query crime indicators for the postcode's datazone using statistics.gov.scot SPARQL.
    Returns a dict of crime metrics or None.
    """
    if not postcode:
        return None

    # Step 1: Convert postcode → datazone
    dz_url = f"https://statistics.gov.scot/sparql"
    dz_query = f"""
    PREFIX qb: <http://purl.org/linked-data/cube#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?datazone
    WHERE {{
        ?s rdfs:label "{postcode}"@en .
        ?s <http://statistics.gov.scot/def/postcode/dataZone> ?datazone .
    }}
    """

    dz_resp = requests.get(dz_url, params={"query": dz_query, "format": "json"})
    dz_data = dz_resp.json()

    try:
        datazone = dz_data["results"]["bindings"][0]["datazone"]["value"]
    except:
        return None

    # Step 2: Query crime indicators for that datazone
    crime_query = f"""
    PREFIX qb: <http://purl.org/linked-data/cube#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?indicator ?value
    WHERE {{
        ?obs qb:dataSet <http://statistics.gov.scot/data/crime> ;
             <http://statistics.gov.scot/def/crime/dataZone> <{datazone}> ;
             <http://statistics.gov.scot/def/crime/indicator> ?indicator ;
             <http://statistics.gov.scot/def/crime/value> ?value .
    }}
    """

    crime_resp = requests.get(dz_url, params={"query": crime_query, "format": "json"})
    crime_data = crime_resp.json()

    results = {}
    for row in crime_data["results"]["bindings"]:
        indicator = row["indicator"]["value"].split("/")[-1]
        value = row["value"]["value"]
        results[indicator] = value

    return results

def extract_tenure(detail_soup):
    """Extract tenure from the property detail page."""
    metrics = detail_soup.select(".pd-metric")
    for m in metrics:
        strong = m.find("strong")
        if not strong:
            continue
        value = strong.get_text(strip=True)

        # The label is hidden in CSS ::before, so detect by icon class
        span = m.find("span", class_="icon-key")
        if span:
            return value  # e.g., "Freehold"

    return None

def fetch_detail_page(url):
    """Download the full property detail page."""
    try:
        r = requests.get(url, headers=HEADERS)
        r.raise_for_status()
        return BeautifulSoup(r.text, "html.parser")
    except:
        return None

def extract_full_description(detail_soup):
    """Extract full property description from the detail page."""
    if not detail_soup:
        return None

    # ESPC uses .itemDescription or .description
    desc_tag = detail_soup.select_one(".itemDescription, .description")
    if desc_tag:
        return desc_tag.get_text(" ", strip=True)

    return None

def parse_listing(card):
    text = card.get_text(" ", strip=True)

    # Property type + address
    if ":" in text:
        prop_type, address = text.split(":", 1)
        prop_type = prop_type.strip()
        address = address.strip()
    else:
        prop_type = None
        address = text

    postcode = extract_postcode(address)

    # Price
    price_tag = card.find(string=lambda s: s and "£" in s)
    price = None
    if price_tag:
        raw = (
            price_tag.replace("£", "")
            .replace(",", "")
            .replace("Fixed Price", "")
            .replace("Offers Over", "")
            .strip()
        )
        try:
            price = int(raw)
        except:
            price = None

    # Beds / baths / receptions
    beds, baths, receptions = extract_facilities(card)

    # URL
    link = card.find("a")
    url = "https://espc.com" + link["href"] if link else None

    # -----------------------------
    # SEARCH-PAGE DESCRIPTION (short)
    # -----------------------------
    search_desc_tag = card.select_one(".description, .itemDescription")
    search_description = (
        search_desc_tag.get_text(" ", strip=True)
        if search_desc_tag else None
    )

    # -----------------------------
    # FULL DESCRIPTION (placeholder)
    # -----------------------------
    full_description = None
    tenure = None

    return {
        "property_type": prop_type,
        "address": address,
        "postcode": postcode,
        "price": price,
        "beds": beds,
        "baths": baths,
        "receptions": receptions,
        "url": url,

        # NEW FIELDS
        "search_description": search_description,
        "full_description": full_description,   
        "tenure": tenure,                       
    }

def filter_listing(item):
    """2+ bed, 2+ bath, £200k–£350k."""
    if item["beds"] is None or item["beds"] < 2:
        return False
    if item["baths"] is None or item["baths"] < 2:
        return False
    if item["price"] is None:
        return False
    if item["price"] < 200000:
        return False
    if item["price"] > 350000:
        return False
    return True

def parse_page(soup):
    """Extract all listing cards from a page."""
    cards = soup.select(".propertyWrap, .property-card, article")
    return [parse_listing(card) for card in cards]

def scrape_espc(max_pages=91):
    """Scrape all pages with correct pagination."""
    filtered = []
    for page in range(1, max_pages + 1):
        print(f"Scraping page {page}...")
        soup = fetch_page(page)
        listings = parse_page(soup)
        for item in listings:
            if filter_listing(item):
                filtered.append(item)
        time.sleep(1)
    return filtered

def save_to_csv(data, filename="espc_filtered.csv"):
    """Save results to CSV."""
    fields = [
        "property_type", "address", "postcode","price", "beds", "baths", "receptions",
        "url","search_description", "full_description", "tenure"
    ]


    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(data)


In [19]:
if __name__ == "__main__":
    results = scrape_espc(max_pages=1)
    print(results)

Scraping page 1...
[{'property_type': 'New Video Offers Over £275,000 2 2 1 New Video Top Floor Flat', 'address': '6/6 Barnie Terrace , Portobello , EH15 1BU A bright and spacious two-bedroom top-floor flat, ideally situated in the popular seaside area of Portobello.\r\n\r\nJust a short five-minute drive from... Watchlist', 'postcode': 'EH15 1BU', 'price': 275000, 'beds': 2, 'baths': 2, 'receptions': 1, 'url': 'https://espc.com/property/6-6-barnie-terrace-portobello-eh15-1bu/36405494?cs=1&sid=902496&p=1', 'search_description': 'A bright and spacious two-bedroom top-floor flat, ideally situated in the popular seaside area of Portobello.\r\n\r\nJust a short five-minute drive from...', 'full_description': None, 'tenure': None}, {'property_type': 'New Exclusive Video Offers Over £265,000 3 2 1 New Exclusive Video Top Floor Flat', 'address': "1/6 East Pilton Farm Place , Fettes , Edinburgh , EH5 2QH Set on the second/top floor of a modern development, with communal gardens and residents' pa

In [ ]:
filtered = scrape_espc(max_pages=192)
save_to_csv(filtered, "espc_basic.csv")

Scraping page 1...


## Get Crime stats for scotland (statistics.gov.scot)

In [ ]:
# Get the datazone for a given postcode using statistics.gov.scot SPARQL endpoint

from io import StringIO

BASE = "https://www.doogal.co.uk/UKPostcodes?Search=EH&page={}"

all_pages = []

for page in range(1, 171):

    url = BASE.format(page)

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        # Read tables directly from HTML
        tables = pd.read_html(response.text)

        # Find the postcode table
        df_page = next(
            table for table in tables
            if "Postcode" in table.columns
            and "Latitude" in table.columns
            and "Active?" in table.columns
        )

        # Keep active postcodes only
        df_page = df_page[df_page["Active?"] == "Yes"]

        all_pages.append(df_page)

        #print(f"Page {page}/170: {len(df_page)} active postcodes")

        sleep(0.5)

    except Exception as e:
        print(f"Page {page} failed: {e}")

# Combine all pages
df = pd.concat(all_pages, ignore_index=True)

# Rename columns
df = df.rename(columns={
    "Postcode": "postcode",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Easting": "easting",
    "Northing": "northing",
    "Grid reference": "gridref",
    "Active?": "active"
})

df.to_csv("edinburgh_postcodes.csv", index=False)

print(f"\nTotal active postcodes: {len(df)}")
df.head()

Page 1/170: 118 active postcodes
Page 2/170: 72 active postcodes
Page 3/170: 82 active postcodes
Page 4/170: 77 active postcodes
Page 5/170: 178 active postcodes
Page 6/170: 132 active postcodes
Page 7/170: 143 active postcodes
Page 8/170: 174 active postcodes
Page 9/170: 155 active postcodes
Page 10/170: 114 active postcodes
Page 11/170: 178 active postcodes
Page 12/170: 131 active postcodes
Page 13/170: 127 active postcodes
Page 14/170: 193 active postcodes
Page 15/170: 115 active postcodes
Page 16/170: 144 active postcodes
Page 17/170: 78 active postcodes
Page 18/170: 59 active postcodes
Page 19/170: 159 active postcodes
Page 20/170: 134 active postcodes
Page 21/170: 154 active postcodes
Page 22/170: 191 active postcodes
Page 23/170: 158 active postcodes
Page 24/170: 161 active postcodes
Page 25/170: 130 active postcodes
Page 26/170: 154 active postcodes
Page 27/170: 159 active postcodes
Page 28/170: 169 active postcodes
Page 29/170: 160 active postcodes
Page 30/170: 178 active post

,postcode,latitude,longitude,easting,northing,gridref,active
0,EH1 1AD,55.948894,-3.192590,325621,673515,NT256735,Yes
1,EH1 1AE,55.949069,-3.189409,325820,673532,NT258735,Yes
2,EH1 1AG,55.947170,-3.189735,325796,673321,NT257733,Yes
3,EH1 1BB,55.951957,-3.189945,325792,673854,NT257738,Yes
4,EH1 1BE,55.951957,-3.189945,325792,673854,NT257738,Yes


In [17]:

SPARQL_URL = "https://statistics.gov.scot/sparql.json"

query = """
PREFIX qb: <http://purl.org/linked-data/cube#>
PREFIX sdmx: <http://purl.org/linked-data/sdmx/2009/dimension#>

SELECT DISTINCT ?area
WHERE {
    ?obs qb:dataSet <http://statistics.gov.scot/data/recorded-crime> ;
         sdmx:refArea ?area .
}
LIMIT 20
"""

r = requests.post(
    SPARQL_URL,
    data={"query": query},
    timeout=30
)

r.raise_for_status()

rows = r.json()["results"]["bindings"]

for row in rows:
    print(row["area"]["value"])

http://statistics.gov.scot/id/statistical-geography/S12000005
http://statistics.gov.scot/id/statistical-geography/S12000006
http://statistics.gov.scot/id/statistical-geography/S12000008
http://statistics.gov.scot/id/statistical-geography/S12000010
http://statistics.gov.scot/id/statistical-geography/S12000011
http://statistics.gov.scot/id/statistical-geography/S12000013
http://statistics.gov.scot/id/statistical-geography/S12000014
http://statistics.gov.scot/id/statistical-geography/S12000017
http://statistics.gov.scot/id/statistical-geography/S12000018
http://statistics.gov.scot/id/statistical-geography/S12000019
http://statistics.gov.scot/id/statistical-geography/S12000020
http://statistics.gov.scot/id/statistical-geography/S12000021
http://statistics.gov.scot/id/statistical-geography/S12000023
http://statistics.gov.scot/id/statistical-geography/S12000026
http://statistics.gov.scot/id/statistical-geography/S12000027
http://statistics.gov.scot/id/statistical-geography/S12000028
http://s

In [18]:
query = """
PREFIX qb: <http://purl.org/linked-data/cube#>

SELECT ?p ?o
WHERE {
    ?obs qb:dataSet <http://statistics.gov.scot/data/recorded-crime> ;
         ?p ?o .
}
LIMIT 50
"""

r = requests.post(
    SPARQL_URL,
    data={"query": query},
    timeout=30
)

r.raise_for_status()

rows = r.json()["results"]["bindings"]

for row in rows:
    print(
        row["p"]["value"],
        "->",
        row["o"]["value"]
    )

http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> http://purl.org/linked-data/cube#Observation
http://statistics.gov.scot/def/measure-properties/count -> 2852
http://purl.org/linked-data/sdmx/2009/dimension#refArea -> http://statistics.gov.scot/id/statistical-geography/S12000005
http://purl.org/linked-data/sdmx/2009/attribute#unitMeasure -> http://statistics.gov.scot/def/concept/measure-units/crimes-or-offences-recorded
http://purl.org/linked-data/sdmx/2009/dimension#refPeriod -> http://reference.data.gov.uk/id/government-year/1996-1997
http://purl.org/linked-data/cube#measureType -> http://statistics.gov.scot/def/measure-properties/count
http://purl.org/linked-data/cube#dataSet -> http://statistics.gov.scot/data/recorded-crime
http://statistics.gov.scot/def/dimension/crimeOrOffence -> http://statistics.gov.scot/def/concept/crime-or-offence/all-crimes
http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> http://purl.org/linked-data/cube#Observation
http://statistics.gov.scot/def/measure-

In [19]:
# grab the datazone for a given postcode using statistics.gov.scot SPARQL endpoint

SPARQL_URL = "https://statistics.gov.scot/sparql.json"

def get_crime_stats():

    query = """
    PREFIX qb: <http://purl.org/linked-data/cube#>
    PREFIX sdmx: <http://purl.org/linked-data/sdmx/2009/dimension#>

    SELECT ?period ?measure ?value
    WHERE {
        ?obs qb:dataSet <http://statistics.gov.scot/data/recorded-crime> ;
             sdmx:refArea
             <http://statistics.gov.scot/id/statistical-geography/S12000036> ;
             sdmx:refPeriod ?period ;
             <http://statistics.gov.scot/def/dimension/crimeOrOffence>
             <http://statistics.gov.scot/def/concept/crime-or-offence/all-crimes> ;
             qb:measureType ?measure ;
             ?measure ?value .
    }
    ORDER BY DESC(?period)
    """

    response = requests.post(
        SPARQL_URL,
        data={"query": query},
        timeout=30
    )

    response.raise_for_status()

    rows = response.json()["results"]["bindings"]

    return [
        {
            "year": x["period"]["value"].split("/")[-1],
            "measure": x["measure"]["value"].split("/")[-1],
            "value": float(x["value"]["value"])
        }
        for x in rows
    ]

In [20]:
crime_stats = get_crime_stats()

crime_stats[:10]

[{'year': '2025-2026', 'measure': 'count', 'value': 39483.0},
 {'year': '2025-2026', 'measure': 'ratio', 'value': 744.0},
 {'year': '2024-2025', 'measure': 'count', 'value': 38116.0},
 {'year': '2024-2025', 'measure': 'ratio', 'value': 718.0},
 {'year': '2023-2024', 'measure': 'ratio', 'value': 667.0},
 {'year': '2023-2024', 'measure': 'count', 'value': 34976.0},
 {'year': '2022-2023', 'measure': 'count', 'value': 32175.0},
 {'year': '2022-2023', 'measure': 'ratio', 'value': 625.0},
 {'year': '2021-2022', 'measure': 'count', 'value': 32032.0},
 {'year': '2021-2022', 'measure': 'ratio', 'value': 633.0}]

## neigbourhood crime rate

In [19]:
# get the postcode lookup data from the Scottish Government website

page_url = "https://www.gov.scot/publications/scottish-index-of-multiple-deprivation-2020v2-postcode-look-up/"

# Find Excel file
r = requests.get(page_url)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

link = soup.find(
    "a",
    href=lambda x: x and ".xlsx" in x.lower()
)

file_url = link["href"]

if file_url.startswith("/"):
    file_url = "https://www.gov.scot" + file_url

# Download workbook
r = requests.get(file_url)
r.raise_for_status()

excel = pd.ExcelFile(BytesIO(r.content))
print(excel.sheet_names)

['SIMD2020 Postcode Lookup', 'All postcodes']


In [20]:
# Read ONLY AllPostcodes sheet
postcode_lookup = pd.read_excel(
    BytesIO(r.content),
    sheet_name="All postcodes"
)

# Download workbook
r = requests.get(file_url)
r.raise_for_status()

print(postcode_lookup.shape)
postcode_lookup.head()

(227066, 6)


,Postcode,DZ,SIMD2020_Rank,SIMD2020_Vigintile,SIMD2020_Decile,SIMD2020_Quintile
0,AB1 0AA,S01006514,6715,20,10,5
1,AB1 0AB,S01006514,6715,20,10,5
2,AB1 0AD,S01006514,6715,20,10,5
3,AB1 0AE,S01006853,5069,15,8,4
4,AB1 0AF,S01006511,6253,18,9,5


In [22]:
# Get the SIMD indicator data from the Scottish Government website
page_url = "https://www.gov.scot/publications/scottish-index-of-multiple-deprivation-2020v2-indicator-data/"

# Find Excel download
r = requests.get(page_url)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

link = soup.find(
    "a",
    href=lambda x: x and ".xlsx" in x.lower()
)

file_url = link["href"]

if file_url.startswith("/"):
    file_url = "https://www.gov.scot" + file_url

# Download workbook
s = requests.get(file_url)
s.raise_for_status()

excel = pd.ExcelFile(BytesIO(s.content))

print(excel.sheet_names)


['Contents & notes', 'Indicator descriptions', 'Data']


In [25]:
# Download workbook
simd = pd.read_excel(
    BytesIO(s.content),
    sheet_name="Data"
)

print(simd.shape)
print(simd.columns.tolist())

simd.head()

(6976, 37)
['Data_Zone', 'Intermediate_Zone', 'Council_area', 'Total_population', 'Working_age_population', 'Income_rate', 'Income_count', 'Employment_rate', 'Employment_count', 'CIF', 'ALCOHOL', 'DRUG', 'SMR', 'DEPRESS', 'LBWT', 'EMERG', 'Attendance', 'Attainment', 'no_qualifications', 'not_participating', 'University', 'drive_petrol', 'drive_GP', 'drive_post', 'drive_primary', 'drive_retail', 'drive_secondary', 'PT_GP', 'PT_post', 'PT_retail', 'Broadband', 'crime_count', 'crime_rate', 'overcrowded_count', 'nocentralheat_count', 'overcrowded_rate', 'nocentralheat_rate']


,Data_Zone,Intermediate_Zone,Council_area,Total_population,Working_age_population,Income_rate,Income_count,Employment_rate,Employment_count,CIF,...,PT_GP,PT_post,PT_retail,Broadband,crime_count,crime_rate,overcrowded_count,nocentralheat_count,overcrowded_rate,nocentralheat_rate
0,S01006506,Culter,Aberdeen City,894,580,0.08,71,0.08,49,65,...,8.863589,5.856135,6.023406,0.105051,11.139188,124.59942,87,10,0.102113,0.011737
1,S01006507,Culter,Aberdeen City,793,470,0.05,43,0.05,25,45,...,9.978272,7.515000,7.926029,0.013587,10.126535,127.69905,85,4,0.101675,0.004785
2,S01006508,Culter,Aberdeen City,624,461,0.06,40,0.04,19,45,...,8.620700,4.321493,5.770910,0.005634,8.101228,129.827368,31,8,0.048212,0.012442
3,S01006509,Culter,Aberdeen City,537,307,0.1,52,0.08,26,80,...,7.935112,8.433328,8.329819,0.113074,4.050614,75.430426,42,6,0.072414,0.010345
4,S01006510,Culter,Aberdeen City,663,415,0.1,68,0.08,32,95,...,5.568964,6.966429,6.632609,0.003096,11.139188,168.011888,50,7,0.086655,0.012132


In [27]:
# Keep only the columns needed and merge with postcode lookup

crime_df = simd[
    [
        "Data_Zone",
        "Total_population",
        "Working_age_population",
        "crime_count",
        "crime_rate"
    ]
].copy()

# Merge with postcode lookup
all_postcodes = postcode_lookup.merge(
    crime_df,
    left_on="DZ",
    right_on="Data_Zone",
    how="left"
)

# Remove duplicate Data Zone column
all_postcodes.drop(columns="Data_Zone", inplace=True)

all_postcodes.head()

,Postcode,DZ,SIMD2020_Rank,SIMD2020_Vigintile,SIMD2020_Decile,SIMD2020_Quintile,Total_population,Working_age_population,crime_count,crime_rate
0,AB1 0AA,S01006514,6715,20,10,5,1123,709,9.113881,81.156556
1,AB1 0AB,S01006514,6715,20,10,5,1123,709,9.113881,81.156556
2,AB1 0AD,S01006514,6715,20,10,5,1123,709,9.113881,81.156556
3,AB1 0AE,S01006853,5069,15,8,4,929,608,14.278747,153.700185
4,AB1 0AF,S01006511,6253,18,9,5,759,453,*,*


In [ ]:
# filter for Edinburgh postcodes (EH prefix) and create a new DataFrame
edinburgh_postcodes = all_postcodes[
    all_postcodes["Postcode"].str.startswith("EH", na=False)
].copy()

edinburgh_postcodes.head()

,Postcode,DZ,SIMD2020_Rank,SIMD2020_Vigintile,SIMD2020_Decile,SIMD2020_Quintile,Total_population,Working_age_population,crime_count,crime_rate
58074,EH1 1AA,S01008676,3285,10,5,3,705,604,264.406223,3750.442883
58075,EH1 1AD,S01008674,4105,12,6,3,1221,1161,220.845045,1808.722725
58076,EH1 1AE,S01008678,3815,11,6,3,1456,1390,184.375221,1266.31333
58077,EH1 1AF,S01008806,4948,15,8,4,1073,929,41.535077,387.092984
58078,EH1 1AL,S01008676,3285,10,5,3,705,604,264.406223,3750.442883


In [46]:
# read the filtered ESPC data from CSV and merge with Edinburgh postcodes to get crime data
properties = pd.read_csv("espc_filtered.csv")
properties["postcode"] = properties["postcode"].str.upper().str.replace(" ", "", regex=False)
edinburgh_postcodes["Postcode"] = (   edinburgh_postcodes["Postcode"]  .str.upper() .str.replace(" ", "", regex=False))

# Add Data Zone + SIMD information
properties = properties.merge(
    edinburgh_postcodes,
    left_on="postcode",
    right_on="Postcode",
    how="left"
)
properties.to_csv("espc_with_crime_data.csv", index=False)
properties.head()


,property_type,address,postcode,price,beds,baths,receptions,url,search_description,Postcode,DZ,SIMD2020_Rank,SIMD2020_Vigintile,SIMD2020_Decile,SIMD2020_Quintile,Total_population,Working_age_population,crime_count,crime_rate
0,"New Exclusive Fixed Price £220,000 2 2 1 New E...","Flat 28 , 3 Salamander Court , Leith , EH6 7JE...",EH67JE,220000,2,2,1,https://espc.com/property/flat-28-3-salamander...,McEwan Fraser Legal is delighted to present th...,EH67JE,S01008781,2020.0,6.0,3.0,2.0,1113.0,906.0,31.404571,282.161461
1,"New Exclusive Virtual Tour Offers Over £240,00...","17/6 King Street , Leith , Edinburgh , EH6 6TQ...",EH66TQ,240000,2,2,1,https://espc.com/property/17-6-king-street-lei...,"Forming part of an exclusive development, this...",EH66TQ,S01008788,4160.0,12.0,6.0,3.0,774.0,644.0,40.522027,523.540395
2,"New Exclusive Video Offers Over £340,000 2 2 2...","30/8 Blackwood Crescent , Newington , Edinburg...",EH91QX,340000,2,2,2,https://espc.com/property/30-8-blackwood-cresc...,"Simply stunning two-bedroom, double upper styl...",EH91QX,S01008668,6043.0,18.0,9.0,5.0,665.0,598.0,9.117456,137.104601
3,"Featured Virtual Tour Offers Over £255,000 2 2...","Flat 9 , 5 Waterfront Avenue , Edinburgh , EH5...",EH51RT,255000,2,2,1,https://espc.com/property/flat-9-5-waterfront-...,Set on the first floor of a desirable resident...,EH51RT,S01008920,4594.0,14.0,7.0,4.0,1366.0,1080.0,48.626432,355.976807
4,"Video Offers Over £350,000 3 2 1 Video Ground ...","54 (flat 2) , Stanley Place , Abbeyhill , Edin...",EH75TB,350000,3,2,1,https://espc.com/property/54-flat-2-stanley-pl...,Seldom available 3 bed elevated ground floor a...,EH75TB,S01008688,3321.0,10.0,5.0,3.0,948.0,716.0,22.287115,235.096146


### for partial postcodes

In [38]:
# Create postcode district: EH7 4AQ -> EH7

# Convert crime rate to numeric
all_postcodes["crime_rate"] = pd.to_numeric(
    all_postcodes["crime_rate"],
    errors="coerce"
)

# Create postcode area: EH1 1AA -> EH1
all_postcodes["postcode_area"] = (
    all_postcodes["Postcode"]
    .str.split()
    .str[0]
)

# Average crime rate by postcode area
crime_by_postcode = (
    all_postcodes[
        ["postcode_area", "DZ", "crime_rate"]
    ]
    .dropna(subset=["crime_rate"])
    .drop_duplicates(["postcode_area", "DZ"])
    .groupby("postcode_area", as_index=False)
    .agg(
        avg_crime_rate=("crime_rate", "mean"),
        min_crime_rate=("crime_rate", "min"),
        max_crime_rate=("crime_rate", "max"),
        datazones=("DZ", "nunique")
    )
)

crime_by_postcode[
    ["avg_crime_rate", "min_crime_rate", "max_crime_rate"]
] = crime_by_postcode[
    ["avg_crime_rate", "min_crime_rate", "max_crime_rate"]
].round(2)

crime_by_postcode.head()

,postcode_area,avg_crime_rate,min_crime_rate,max_crime_rate,datazones
0,AB1,408.08,23.68,4351.50,124
1,AB10,609.67,42.68,4351.50,42
2,AB11,930.97,74.15,4351.50,35
3,AB12,227.93,45.41,1633.77,33
4,AB13,173.01,81.16,309.06,3


In [40]:
crime_by_postcode = (
    all_postcodes[
        ["postcode_area", "DZ", "crime_rate"]
    ]
    .dropna()
    .drop_duplicates(["postcode_area", "DZ"])
    .groupby("postcode_area", as_index=False)
    .agg(
        crime_rate=("crime_rate", "mean"),
        datazones=("DZ", "nunique")
    )
)

crime_by_postcode["crime_rate"] = crime_by_postcode["crime_rate"].round(2)
crime_by_postcode

,postcode_area,crime_rate,datazones
0,AB1,408.08,124
1,AB10,609.67,42
2,AB11,930.97,35
3,AB12,227.93,33
4,AB13,173.01,3
...,...,...,...
467,TD8,195.56,8
468,TD9,255.05,23
469,ZE1,305.30,14
470,ZE2,89.53,12


In [42]:
# property dataframe

df = pd.read_excel("zoopla_property_details.xlsx")

df["bathrooms"] = pd.to_numeric(df["bathrooms"], errors="coerce")
df["bedrooms"] = pd.to_numeric(df["bedrooms"], errors="coerce")

# 2+ bathrooms and maximum 3 bedrooms
properties = df[
    (df["bathrooms"] >= 2) &
    (df["bedrooms"] <= 3)
].copy()

print(f"Properties after filter: {len(properties)}")


Properties after filter: 300


In [44]:
df = df.merge(
    crime_by_postcode,
    left_on="postcode",
    right_on="postcode_area",
    how="left"
)

df.drop(columns="postcode_area", inplace=True)
df.head()
df.to_csv("zoopla_properties_with_crime.csv", index=False)